In [1]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
from torchvision import models, transforms

In [2]:
import os

base = "/kaggle/input/best-alzheimer-mri-dataset-99-accuracy"

for root, dirs, files in os.walk(base):
    print(root)
    break

In [3]:
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [5]:
dataset_path = "/kaggle/input/datasets/lukechugh/best-alzheimer-mri-dataset-99-accuracy/Combined Dataset"

In [6]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

In [7]:
class AlzheimerDataset(data.Dataset):

    def __init__(self, path, train=True, transform=None):
        self.transform = transform
        self.files = []
        self.labels = []

        folder = "train" if train else "test"
        path = os.path.join(path, folder)

        self.classes = {
            "No Impairment":0,
            "Very Mild Impairment":1,
            "Mild Impairment":2,
            "Moderate Impairment":3
        }

        for class_name,label in self.classes.items():
            class_path = os.path.join(path,class_name)

            if os.path.exists(class_path):

                for img in os.listdir(class_path):

                    if img.endswith(".jpg"):
                        self.files.append(os.path.join(class_path,img))
                        self.labels.append(label)

        print("Total images:",len(self.files))

    def __len__(self):
        return len(self.files)

    def __getitem__(self,idx):

        img = Image.open(self.files[idx]).convert("L")

        if self.transform:
            img = self.transform(img)

        label = self.labels[idx]

        return img,label

In [8]:
train_dataset = AlzheimerDataset(dataset_path,train=True,transform=transform)
test_dataset  = AlzheimerDataset(dataset_path,train=False,transform=transform)

Total images: 10240
Total images: 1279


In [9]:
train_loader = data.DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2
)

test_loader = data.DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [10]:
model = models.resnet18(pretrained=True)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 135MB/s]


In [11]:
model.conv1 = nn.Conv2d(
    3,64,kernel_size=7,stride=2,padding=3,bias=False
)

In [12]:
model.fc = nn.Sequential(
    nn.Linear(512,256),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(256,4)
)

In [13]:
model = model.to(device)

In [14]:
for param in model.parameters():
    param.requires_grad = False

for param in model.layer4.parameters():
    param.requires_grad = True

for param in model.fc.parameters():
    param.requires_grad = True

In [15]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam([
    {"params": model.layer4.parameters(), "lr": 1e-4},
    {"params": model.fc.parameters(), "lr": 1e-3}
])

In [16]:
epochs = 10

for epoch in range(epochs):

    model.train()
    running_loss = 0

    loop = tqdm(train_loader)

    for images,labels in loop:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        loss = criterion(outputs,labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        loop.set_description(f"Epoch [{epoch+1}/{epochs}]")
        loop.set_postfix(loss=loss.item())

    print("Epoch Loss:", running_loss/len(train_loader))

Epoch [1/10]: 100%|██████████| 320/320 [00:57<00:00,  5.60it/s, loss=0.189]


Epoch Loss: 0.49786173105239867


Epoch [2/10]: 100%|██████████| 320/320 [00:21<00:00, 14.71it/s, loss=0.218]


Epoch Loss: 0.2071244894817937


Epoch [3/10]: 100%|██████████| 320/320 [00:23<00:00, 13.54it/s, loss=0.0325]


...

Epoch Loss: 0.030199480435487657


Epoch [9/10]: 100%|██████████| 320/320 [00:22<00:00, 14.21it/s, loss=0.00516]


Epoch Loss: 0.038126756385042884


Epoch [10/10]: 100%|██████████| 320/320 [00:21<00:00, 14.82it/s, loss=0.0371]

Epoch Loss: 0.030827314754338885


In [17]:
torch.save(model.state_dict(),"alzheimer_resnet18.pth")

In [18]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images,labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _,predicted = torch.max(outputs,1)

        total += labels.size(0)
        correct += (predicted==labels).sum().item()

accuracy = 100 * correct / total

print("Test Accuracy:",accuracy)

Test Accuracy: 94.68334636434714
